<a href="https://colab.research.google.com/github/EmanSayedSeddik/ManufacturingDownTime/blob/main/Work%20Force%20Assessment%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

hr = "HR_Tables_DIRTY.xlsx"
read = lambda sheet: pd.read_excel(hr, sheet_name=sheet, dtype=str, keep_default_na=False)
ros, emp = read("Fact_Shift_Roster"), read("Dim_Employee")

# 1) fix machine_id typos (MCH-O1234, mch-01234, MCH-1234 -> MCH-01234); IDs that still do not exist become blank
valid = {f"MCH-{i:05d}" for i in range(1, 5001)}
m = ros.machine_id.str.upper().str.replace("O", "0")
m = "MCH-" + m.str[4:].str.zfill(5)
ros["machine_id"] = m.where(m.isin(valid), "")

# 2) drop roster rows dated after the employee's termination date
term = emp.drop_duplicates("employee_id").set_index("employee_id").termination_date
t = ros.employee_id.map(term)
ros = ros[~((t != "") & (ros.work_date > t))]

# 3) rebuild the assignments = consecutive working days on the same machine
a = ros[ros.machine_id != ""].sort_values(["employee_id", "work_date"])
run = ((a.machine_id != a.machine_id.shift()) | (a.employee_id != a.employee_id.shift())).cumsum()
asg = a.groupby(run).agg(employee_id=("employee_id", "first"), machine_id=("machine_id", "first"), factory_id=("factory_id", "first"),
                         assignment_start_date=("work_date", "min"), assignment_end_date=("work_date", "max")).reset_index(drop=True)
asg["machine_assignment_count"] = asg.groupby("employee_id").cumcount() + 1
asg["previous_machine_id"] = asg.groupby("employee_id").machine_id.shift().fillna("")
asg["is_machine_rotation"] = (asg.previous_machine_id != "").map({True: "Yes", False: "No"})
asg.insert(0, "assignment_id", ["ASG-" + str(i + 1).zfill(6) for i in range(len(asg))])
asg.to_csv("Fact_Workforce_Assignment_clean.csv", index=False)

## Workforce Assignment ETL & Data Cleaning Pipeline

### Overview
This Python script cleans raw shift roster records and reconstructs the **`Fact_Workforce_Assignment`** fact table from `HR_Tables_DIRTY.xlsx`. Using vectorized `pandas` operations, it standardizes machine identifiers, enforces HR termination date rules, and aggregates consecutive daily shifts into continuous machine assignment intervals.

---

### Input & Output
* **Input File:** `HR_Tables_DIRTY.xlsx`
  * `Fact_Shift_Roster`: Daily employee shift logs and machine assignments.
  * `Dim_Employee`: Employee master data including `termination_date`.
* **Output File:** `Fact_Workforce_Assignment_clean.csv` (cleaned and reconstructed assignment table).

---

### Data Cleaning & Transformation Steps

1. **Raw Data Ingestion (String-Safe Loading)**
   * Reads `Fact_Shift_Roster` and `Dim_Employee` with `dtype=str` and `keep_default_na=False` to preserve leading zeros and prevent unintended type coercion during string standardization.

2. **Machine ID Standardization & Validation**
   * Builds a reference set of valid machine IDs (`MCH-00001` through `MCH-05000`).
   * Cleans formatting typos in `machine_id` by converting text to uppercase, replacing letter `"O"` with `"0"`, and zero-padding numeric suffixes to 5 digits (`MCH-XXXXX`).
   * Nullifies (blanks out) any invalid or out-of-range machine IDs.

3. **Post-Termination Record Filtering**
   * Deduplicates `Dim_Employee` on `employee_id` to map each employee's `termination_date`.
   * Removes invalid shift roster entries where `work_date` occurs after the employee's recorded `termination_date`.

4. **Consecutive Assignment Reconstruction & Feature Engineering**
   * Filters out unassigned shifts (`machine_id != ""`) and sorts records chronologically by `employee_id` and `work_date`.
   * Uses shift comparison and cumulative sum (`cumsum()`) to detect contiguous blocks of working days on the same machine per employee.
   * Aggregates each contiguous block to determine `assignment_start_date` (`min`) and `assignment_end_date` (`max`).
   * Generates tracking attributes per employee:
     * `machine_assignment_count`: Sequential assignment number for each employee.
     * `previous_machine_id`: The machine ID from the employee's immediately preceding assignment.
     * `is_machine_rotation`: Flags `"Yes"` if the assignment represents a rotation from a previous machine, and `"No"` otherwise.
   * Assigns formatted primary keys (`ASG-000001`, `ASG-000002`, ...) and exports the final dataset to CSV.